In [1]:
# uv add transformers "datasets[audio]" accelerate

In [2]:
import os

os.environ["PATH"] += os.pathsep + r"C:\ffmpeg\bin"

## toml에 직접 추가

```
PS C:\Users\user\Documents\GitHub\STUDY\chaewony\ch05> uv add torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index https://download.pytorch.org/whl/cu126
  × No solution found when resolving dependencies for split (markers: python_full_version >= '3.14' and sys_platform
  │ == 'win32'):
  ╰─▶ Because only requests==2.28.1 is available and datasets>=5.0.1 depends on requests>=2.32.2, we can conclude that
      datasets>=5.0.1 cannot be used.
      And because your project depends on datasets[audio]>=5.0.1, we can conclude that your project's requirements
      are unsatisfiable.
```

```
[[tool.uv.index]]
name = "pytorch"
url = "https://download.pytorch.org/whl/cu128"
explicit = true

[tool.uv.sources]
torch = { index = "pytorch" }
torchvision = { index = "pytorch" }
torchaudio = { index = "pytorch" }
```

원인: --index로 PyTorch CUDA 저장소를 지정하니 datasets의 requests까지 PyTorch 저장소에서 찾으려 해서 의존성 충돌이 발생함.

해결: PyTorch CUDA index를 explicit = true로 등록하고 torch 계열만 해당 index를 사용하도록 pyproject.toml에 지정.

In [3]:
# uv add torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0

In [4]:
from pprint import pprint

In [5]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
# from datasets import load_dataset

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,   # 청크별로 타임스탬프 반환
    chunk_length_s=10,  # 입력 오디오 10초씩 나누기r
    stride_length_s=2,  # 2초씩 겹치도록 청크 나누기
) 

# dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
# sample = dataset[0]["audio"]
sample = "audio/lsy_audio_2023_58s.mp3"

result = pipe(sample)
# print(result["text"])

pprint(result)

c:\Users\user\Documents\GitHub\learn_do_it_llm_agent\ch05\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0
c:\Users\user\Documents\GitHub\learn_do_it_llm_agent\ch05\.venv\Lib\site-packages\transformers\models\whisper\generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Passing a tuple of `past_key_values` is deprecated and will be remov

{'chunks': [{'text': ' 안녕하세요. 이 강의는 GPT API로 챗봇 만들기 라는 내용을 다루는 강의입니다.',
             'timestamp': (0.0, 6.3)},
            {'text': ' GPT API에 대해서 생소하신 분들도 있을텐데', 'timestamp': (7.18, 10.0)},
            {'text': ' 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서',
             'timestamp': (11.0, 17.0)},
            {'text': ' 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요.',
             'timestamp': (17.0, 21.0)},
            {'text': ' 그래서 이런 강의들이 사실 많이 있습니다.', 'timestamp': (21.0, 24.0)},
            {'text': ' 그래서 여러 가지들이 있는데 이 강의 특징이라고 한다면',
             'timestamp': (24.0, 27.48)},
            {'text': ' GPT로 명확한 미션을 달성하는', 'timestamp': (27.48, 29.58)},
            {'text': ' 챕터 프로그램을 만드는게 사실', 'timestamp': (29.58, 31.66)},
            {'text': ' 쉽지는 않은데 이걸 어떻게 해서', 'timestamp': (31.66, 34.32)},
            {'text': ' 구현을 하는지 그리고 그게 왜 필요한지에 대해서', 'timestamp': (34.32, 36.4)},
            {'text': ' 좀 이야기를 할 거고요.', 'timestamp': (36.4, 37.36)},
            {'text': ' 그 예제로 예제는 여러가지가 될 수 있는데', 'timestamp

> Whisper + transformers pipeline은 로컬 GPU에서 직접 모델을 돌리는 방식이고, gpt-4o-transcribe-diarize는 OpenAI API에 파일을 보내서 transcription·화자 분리를 하는 방식이라 torch, AutoModelForSpeechSeq2Seq, pipeline() 부분이 필요 없다

In [9]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

audio_file_path = "audio/lsy_audio_2023_58s.mp3"

with open(audio_file_path, "rb") as audio_file:
    result = client.audio.transcriptions.create(
        model="gpt-4o-transcribe-diarize",
        file=audio_file,
        chunking_strategy="auto",
        response_format="diarized_json",
    )

for segment in result.segments:
    print(f"[{segment.start:.2f}s - {segment.end:.2f}s] {segment.speaker}: {segment.text}")

[0.00s - 0.60s] A:  안녕하세요.
[1.05s - 6.20s] A:  이 강의는 GPT API로 챗봇 만들기라는 내용을 다루는 강의입니다.
[7.05s - 10.30s] A:  GPT API에 대해서 생소하신 분들도 있을텐데,
[10.65s - 21.35s] A:  우리가 잘 알고 있는 채 GPT, 채 GPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할거에요.
[21.60s - 23.70s] A:  그래서 이런 강의들이 사실 많이 있습니다.
[23.90s - 26.95s] A:  그래서 여러가지들이 있는데 좀 이 강의의 특징이라고 하는
[26.71s - 37.26s] A:  칭이라고 한다면 GPT로 명확한 미션을 달성하는 챕봇 프로그램을 만드는 게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요
[38.11s - 48.26s] A:  그 예제로 예제는 여러 가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다
[49.49s - 51.89s] A:  그래서 프로그램이 실행되는 모습을 한번 보여드릴게요.
[51.99s - 57.94s] A:  우리가 만들 프로그램은 이런 식으로 나타나게 되고


In [11]:
# chunks를 CSV 파일로 저장
start_end_text = []

for segment in result.segments:
    start_end_text.append([segment.start, segment.end, segment.text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
df.to_csv("lsy_audio_2023_58.csv", index=False, sep="|")
display(df)

,start,end,text
0,0.000,0.600,안녕하세요.
1,1.050,6.200,이 강의는 GPT API로 챗봇 만들기라는 내용을 다루는 강의입니다.
2,7.050,10.300,"GPT API에 대해서 생소하신 분들도 있을텐데,"
3,10.650,21.350,"우리가 잘 알고 있는 채 GPT, 채 GPT 기능을 이용해서 우리가 원하는 프로그..."
4,21.600,23.700,그래서 이런 강의들이 사실 많이 있습니다.
5,23.900,26.950,그래서 여러가지들이 있는데 좀 이 강의의 특징이라고 하는
6,26.708,37.258,칭이라고 한다면 GPT로 명확한 미션을 달성하는 챕봇 프로그램을 만드는 게 사실 ...
7,38.108,48.258,그 예제로 예제는 여러 가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이...
8,49.492,51.892,그래서 프로그램이 실행되는 모습을 한번 보여드릴게요.
9,51.992,57.942,우리가 만들 프로그램은 이런 식으로 나타나게 되고
